# XGBoost Model

Third gradient-boosting model for the EV Purchase Prediction competition.

Current results:

- Logistic Regression CV: **0.938096**
- CatBoost OOF: **0.941711**
- LightGBM OOF: **0.941770**
- CatBoost + LightGBM rank ensemble: **0.941911**

Best public leaderboard score: **0.941690**

Goal: determine whether XGBoost provides useful standalone performance
and, more importantly, prediction diversity for a stronger ensemble.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.shape, test.shape

((668665, 15), (286571, 14))

In [3]:
categorical_features = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

In [4]:
for col in categorical_features:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [5]:
train[categorical_features].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
dtype: object

In [6]:
X = train.drop(columns=["Will_Buy_EV", "id"])

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

X_test = test.drop(columns=["id"])

In [7]:
X.shape, y.shape, X_test.shape

((668665, 13), (668665,), (286571, 13))

In [8]:
X[categorical_features].dtypes

Gender                    category
City_Type                 category
Current_Car_Type          category
Home_Charging_Possible    category
Subsidy_Available         category
Range_Anxiety_Level       category
dtype: object

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
X_train.shape, X_valid.shape

((534932, 13), (133733, 13))

In [11]:
early_stop = xgb.callback.EarlyStopping(
    rounds=150,
    save_best=True
)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",

    n_estimators=3000,
    learning_rate=0.03,

    max_depth=6,
    min_child_weight=5,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    tree_method="hist",
    enable_categorical=True,

    random_state=42,
    n_jobs=-1,

    callbacks=[early_stop]
)

In [12]:
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)]
)

[0]	validation_0-auc:0.93668
[1]	validation_0-auc:0.92740
[2]	validation_0-auc:0.92965
[3]	validation_0-auc:0.93455
[4]	validation_0-auc:0.93658
[5]	validation_0-auc:0.93861
[6]	validation_0-auc:0.93874
[7]	validation_0-auc:0.93880
[8]	validation_0-auc:0.93881
[9]	validation_0-auc:0.93888
[10]	validation_0-auc:0.93894
[11]	validation_0-auc:0.93895
[12]	validation_0-auc:0.93899
[13]	validation_0-auc:0.93901
[14]	validation_0-auc:0.93873
[15]	validation_0-auc:0.93900
[16]	validation_0-auc:0.93904
[17]	validation_0-auc:0.93901
[18]	validation_0-auc:0.93875
[19]	validation_0-auc:0.93840
[20]	validation_0-auc:0.93854
[21]	validation_0-auc:0.93870
[22]	validation_0-auc:0.93882
[23]	validation_0-auc:0.93887
[24]	validation_0-auc:0.93890
[25]	validation_0-auc:0.93911
[26]	validation_0-auc:0.93913
[27]	validation_0-auc:0.93915
[28]	validation_0-auc:0.93917
[29]	validation_0-auc:0.93902
[30]	validation_0-auc:0.93905
[31]	validation_0-auc:0.93907
[32]	validation_0-auc:0.93909
[33]	validation_0-au

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",[<xgboost.call...x702ed8335a60>]
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [13]:
xgb_model.best_iteration

879

In [14]:
xgb_valid_predictions = xgb_model.predict_proba(
    X_valid
)[:, 1]

In [15]:
xgb_auc = roc_auc_score(
    y_valid,
    xgb_valid_predictions
)

xgb_auc

0.9416720313757281

In [16]:
xgb_model.best_iteration

879

In [17]:
xgb_model.best_score

0.9416720313757283

In [18]:
xgb_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

xgb_importance

,Feature,Importance
11,Subsidy_Available,0.752989
6,Environmental_Concern_Level,0.203199
12,Range_Anxiety_Level,0.020979
1,Annual_Income_USD,0.009835
10,Home_Charging_Possible,0.003452
2,Daily_Commute_km,0.001718
0,Age,0.001408
8,City_Type,0.001222
4,Charging_Stations_Near_Home,0.001165
5,Charging_Stations_Near_Work,0.001064


In [19]:
single_split_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "CatBoost",
        "LightGBM",
        "XGBoost"
    ],
    "Single_Split_AUC": [
        0.9379578769,
        0.9414339188,
        0.941669,
        xgb_auc
    ]
})

single_split_comparison.sort_values(
    "Single_Split_AUC",
    ascending=False
)

,Model,Single_Split_AUC
3,XGBoost,0.941672
2,LightGBM,0.941669
1,CatBoost,0.941434
0,Logistic Regression,0.937958


In [20]:
from sklearn.model_selection import StratifiedKFold

In [21]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_xgb = np.zeros(len(X))
test_predictions_xgb = np.zeros(len(X_test))

fold_scores_xgb = []
best_iterations_xgb = []

In [22]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    print(f"\n{'=' * 50}")
    print(f"FOLD {fold}")
    print(f"{'=' * 50}")

    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    early_stop = xgb.callback.EarlyStopping(
        rounds=150,
        save_best=True
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",

        n_estimators=3000,
        learning_rate=0.03,

        max_depth=6,
        min_child_weight=5,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        tree_method="hist",
        enable_categorical=True,

        random_state=42 + fold,
        n_jobs=-1,

        callbacks=[early_stop]
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        verbose=False
    )

    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]

    oof_predictions_xgb[valid_idx] = valid_pred
    test_predictions_xgb += test_pred / 5

    fold_auc = roc_auc_score(
        y_valid_fold,
        valid_pred
    )

    fold_scores_xgb.append(fold_auc)
    best_iterations_xgb.append(
        model.best_iteration
    )

    print(f"Fold {fold} AUC: {fold_auc:.6f}")
    print(f"Best iteration: {model.best_iteration}")


FOLD 1
Fold 1 AUC: 0.940641
Best iteration: 1020

FOLD 2
Fold 2 AUC: 0.941580
Best iteration: 760

FOLD 3
Fold 3 AUC: 0.942841
Best iteration: 808

FOLD 4
Fold 4 AUC: 0.942434
Best iteration: 763

FOLD 5
Fold 5 AUC: 0.941832
Best iteration: 855


In [23]:
oof_auc_xgb = roc_auc_score(
    y,
    oof_predictions_xgb
)

print("\nFold scores:")

for fold, score in enumerate(fold_scores_xgb, start=1):
    print(f"Fold {fold}: {score:.6f}")

print()
print(f"Mean fold AUC: {np.mean(fold_scores_xgb):.6f}")
print(f"OOF AUC:       {oof_auc_xgb:.6f}")
print(f"Std AUC:       {np.std(fold_scores_xgb):.6f}")

print()
print("Best iterations:")
print(best_iterations_xgb)


Fold scores:
Fold 1: 0.940641
Fold 2: 0.941580
Fold 3: 0.942841
Fold 4: 0.942434
Fold 5: 0.941832

Mean fold AUC: 0.941866
OOF AUC:       0.941857
Std AUC:       0.000756

Best iterations:
[1020, 760, 808, 763, 855]


In [24]:
xgb_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "xgboost_oof": oof_predictions_xgb
})

xgb_oof_df.to_csv(
    "../outputs/predictions/xgboost_oof.csv",
    index=False
)

In [25]:
xgb_test_df = pd.DataFrame({
    "id": test["id"],
    "xgboost_prediction": test_predictions_xgb
})

xgb_test_df.to_csv(
    "../outputs/predictions/xgboost_test.csv",
    index=False
)

In [26]:
print(f"XGBoost mean fold AUC: {np.mean(fold_scores_xgb):.6f}")
print(f"XGBoost OOF AUC:       {oof_auc_xgb:.6f}")
print(f"XGBoost std:           {np.std(fold_scores_xgb):.6f}")
print(f"Best iterations:       {best_iterations_xgb}")

XGBoost mean fold AUC: 0.941866
XGBoost OOF AUC:       0.941857
XGBoost std:           0.000756
Best iterations:       [1020, 760, 808, 763, 855]


In [27]:
cat_oof = pd.read_csv(
    "../outputs/predictions/catboost_oof.csv"
)

lgb_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_oof.csv"
)

xgb_oof = pd.read_csv(
    "../outputs/predictions/xgboost_oof.csv"
)

In [28]:
assert cat_oof["id"].equals(lgb_oof["id"])
assert cat_oof["id"].equals(xgb_oof["id"])
assert cat_oof["target"].equals(lgb_oof["target"])
assert cat_oof["target"].equals(xgb_oof["target"])

print("All OOF prediction files aligned.")

All OOF prediction files aligned.


In [29]:
prediction_matrix = pd.DataFrame({
    "CatBoost": cat_oof["catboost_oof"],
    "LightGBM": lgb_oof["lightgbm_oof"],
    "XGBoost": xgb_oof["xgboost_oof"]
})

prediction_matrix.corr()

,CatBoost,LightGBM,XGBoost
CatBoost,1.000000,0.997437,0.997973
LightGBM,0.997437,1.000000,0.997616
XGBoost,0.997973,0.997616,1.000000


In [30]:
cat_rank = cat_oof["catboost_oof"].rank(
    method="average",
    pct=True
)

lgb_rank = lgb_oof["lightgbm_oof"].rank(
    method="average",
    pct=True
)

xgb_rank = xgb_oof["xgboost_oof"].rank(
    method="average",
    pct=True
)

y_ensemble = cat_oof["target"]

In [31]:
three_model_equal = (
    cat_rank
    + lgb_rank
    + xgb_rank
) / 3

equal_three_auc = roc_auc_score(
    y_ensemble,
    three_model_equal
)

equal_three_auc

0.9419817144358864

In [32]:
three_model_results = []

weights = np.arange(
    0.0,
    1.01,
    0.05
)

for cat_w in weights:

    for lgb_w in weights:

        xgb_w = 1.0 - cat_w - lgb_w

        if xgb_w < -1e-9:
            continue

        if xgb_w > 1.0:
            continue

        blended_rank = (
            cat_w * cat_rank
            + lgb_w * lgb_rank
            + xgb_w * xgb_rank
        )

        auc = roc_auc_score(
            y_ensemble,
            blended_rank
        )

        three_model_results.append({
            "CatBoost_Weight": cat_w,
            "LightGBM_Weight": lgb_w,
            "XGBoost_Weight": xgb_w,
            "OOF_AUC": auc
        })

In [33]:
three_model_results = pd.DataFrame(
    three_model_results
)

three_model_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(15)

,CatBoost_Weight,LightGBM_Weight,XGBoost_Weight,OOF_AUC
85,0.20,0.35,0.45,0.941988
102,0.25,0.35,0.40,0.941988
101,0.25,0.30,0.45,0.941987
86,0.20,0.40,0.40,0.941987
84,0.20,0.30,0.50,0.941987
67,0.15,0.35,0.50,0.941986
68,0.15,0.40,0.45,0.941986
117,0.30,0.30,0.40,0.941985
103,0.25,0.40,0.35,0.941985
118,0.30,0.35,0.35,0.941984


In [34]:
best_three_model_blend = (
    three_model_results
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .iloc[0]
)

best_three_model_blend

CatBoost_Weight    0.200000
LightGBM_Weight    0.350000
XGBoost_Weight     0.450000
OOF_AUC            0.941988
Name: 85, dtype: float64

In [35]:
comparison = pd.DataFrame({
    "Model": [
        "CatBoost",
        "LightGBM",
        "XGBoost",
        "CatBoost + LightGBM Rank",
        "Equal 3-Model Rank",
        "Best 3-Model Rank"
    ],
    "OOF_AUC": [
        roc_auc_score(
            y_ensemble,
            cat_oof["catboost_oof"]
        ),
        roc_auc_score(
            y_ensemble,
            lgb_oof["lightgbm_oof"]
        ),
        roc_auc_score(
            y_ensemble,
            xgb_oof["xgboost_oof"]
        ),
        0.941911,
        equal_three_auc,
        best_three_model_blend["OOF_AUC"]
    ]
})

comparison.sort_values(
    "OOF_AUC",
    ascending=False
)

,Model,OOF_AUC
5,Best 3-Model Rank,0.941988
4,Equal 3-Model Rank,0.941982
3,CatBoost + LightGBM Rank,0.941911
2,XGBoost,0.941857
1,LightGBM,0.941770
0,CatBoost,0.941711


In [36]:
cat_test = pd.read_csv(
    "../outputs/predictions/catboost_test.csv"
)

lgb_test = pd.read_csv(
    "../outputs/predictions/lightgbm_test.csv"
)

xgb_test = pd.read_csv(
    "../outputs/predictions/xgboost_test.csv"
)

In [37]:
assert cat_test["id"].equals(lgb_test["id"])
assert cat_test["id"].equals(xgb_test["id"])

print("All test prediction files aligned.")

All test prediction files aligned.


In [38]:
cat_test_rank = cat_test["catboost_prediction"].rank(
    method="average",
    pct=True
)

lgb_test_rank = lgb_test["lightgbm_prediction"].rank(
    method="average",
    pct=True
)

xgb_test_rank = xgb_test["xgboost_prediction"].rank(
    method="average",
    pct=True
)

In [43]:
three_model_test_pred = (
    0.20 * cat_test_rank
    + 0.35 * lgb_test_rank
    + 0.45 * xgb_test_rank
)

In [44]:
submission_004 = pd.DataFrame({
    "id": cat_test["id"],
    "Will_Buy_EV": three_model_test_pred
})

In [40]:
three_model_test_pred.min(), three_model_test_pred.max()

(np.float64(9.910284013385861e-05), np.float64(0.9999975573243629))

In [41]:
three_model_test_pred.min(), three_model_test_pred.max()

(np.float64(9.910284013385861e-05), np.float64(0.9999975573243629))

In [45]:
submission_004.shape

(286571, 2)

In [46]:
submission_004.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [47]:
submission_004["id"].equals(cat_test["id"])

True

In [48]:
submission_004.to_csv(
    "../outputs/submissions/submission_004_three_model_rank_blend.csv",
    index=False
)

# Current Model Leaderboard

| Model | Local OOF / CV ROC-AUC | Public Kaggle ROC-AUC |
|---|---:|---:|
| Constant baseline | 0.500000 | — |
| Logistic Regression | 0.938096 | — |
| CatBoost | 0.941711 | 0.941560 |
| LightGBM | 0.941770 | 0.941650 |
| XGBoost | 0.941857 | — |
| CatBoost + LightGBM Rank | 0.941911 | 0.941690 |
| Three-Model Rank Ensemble | **0.941988** | **0.941710** |

Current best local result:

**0.941988**

Current best public leaderboard score:

**0.941710**